In [6]:
import json
import pandas as pd
import ast
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# GSS (General Social Survey) 2024 Cross-section, Release 3.
# Reads the Stata extract at ../data/GSS2024.dta (one column per GSS variable
# code -- lowercase in the .dta file --, one row per respondent).
personas = pd.read_stata("../data/GSS2024.dta", convert_categoricals=True)

# Allowlist of persona-defining variables, mapped to a human-readable label for
# the LLM prompt. Selected from the full 2024 GSS Release 3 variable list to
# maximize socio-economic and demographic background depth.
#
# Demographic/background facts (citizenship, birthplace, ancestry, migration
# history, language) are INCLUDED even though they are immigration-adjacent --
# they describe who the respondent is, not what they think government should
# do. Only explicit attitude/opinion items are excluded:
#  - immigration/asylum ATTITUDE items (LETIN1A, LETIN1, IMMCRIME, IMMJOBS,
#    IMMFATE, LETINHSP, LETINASN, ADOPTUS, IMMAMECO, IMMASSIM, IMMWLFARE,
#    TOPPROB2_IMMG) -- these ask what the respondent believes policy should be,
#    which is exactly the simulation's stance dimension
#  - race/civil-rights attitude items (RACOPEN*, RACLIVE, RACDIF1-5, WRKWAYUP,
#    WLTHWHTS/BLKS/HSPS, WORKWHTS/BLKS/HSPS, MARWHT/BLK/ASIAN/HISP, LIVEBLKS/WHTS,
#    AFFRMACT, DISCAFF*, HELPBLK, DSNDNTPAY)
#  - gender-role/political items (FECHLD, FEPRESCH, FEFAM, FEPOL*, MEOVRWRK,
#    FEOVRWRK, FEHIRE, FEJOBAFF)
#  - party/vote/political-views/national-spending-priority items (PARTYID,
#    POLVIEWS, VOTE16, VOTE20, PRES16, PRES20, IF16WHO, IF20WHO, NAT*)
#  - institutional confidence/trust items (CON*, TRCONG, TRCOURTS, TRPPL,
#    TRUST*, CANTRUST, FAIR*, HELPFUL*)
#  - civil-liberties (Stouffer) tolerance items (SPKATH/COLATH/LIBATH,
#    SPKRAC/COLRAC/LIBRAC, SPKCOM/COLCOM/LIBCOM, SPKMSLM/COLMSLM/LIBMSLM)
#  - abortion items (AB*)
#  - government-intervention/tax policy items (TAX, EQWLTH, HELPPOOR, HELPNOT,
#    HELPSICK) -- note PARSOL/KIDSSOL/GOODLIFE are KEPT: they ask about personal
#    economic mobility, not government policy
#  - gun/crime/punishment policy items (OWNGUN, ROWNGUN, PISTOL, RIFLE, SHOTGUN,
#    HUNT*, GUNLAW, CAPPUN, GRASS, COURTS*, POL(HITOK|ABUSE|MURDR|ESCAP|ATTAK), FEAR)
#  - sex/marriage-politics items (SEXEDUC, TEENSEX, PREMARSX, PILLOK, MARSAME*,
#    HOMOSEX, XMARSEX, DIVLAW*, PRAYER*, CHNGDOCS, ACPTTG, RELIGINF, SUICIDE*,
#    LETDIE1, XMOVIE*, PORNLAW)
#  - the ANES, ISSP (Digital Societies / National Identity), Adult & Child
#    Mental Health Stigma, High Risk Behaviors, and GSS Next Follow-on topical
#    modules in full (explicitly political, sensitive, or hypothetical-vignette
#    judgments rather than respondent self-description)
#  - technical/paradata/household-roster/interviewer variables (ID, weights,
#    SAMPLE, YEAR, DATEINTV, BALLOT, FORM, INT*, RLOOKS, RGROOMED, RESPOND,
#    CONSENT, MODE, FEEUSED, SPANENG, VPSU, VSTRAT, KISH, DEVTYPE, WHOELSE*,
#    RELATE1-14/GENDER1-14/OLD1-14/MAR1-14/AWAY1-14/WHERE1-14 roster linkage)
#  - the verbal-ability test battery (WORDA-WORDSUM*) -- a cognitive test, not
#    an attitude or demographic
#
# NOTE: INCOME/RINCOME are GSS's legacy historical brackets (top-coded at
# "$25,000 or more", frozen since the 1970s for trend consistency) and are
# useless for describing 2024 households. CONINC/CONRINC (the recoded,
# continuous, constant-dollar equivalents) are used instead.
INCLUDE_VARIABLES = {
    # -- Demographics ------------------------------------------------
    "age":       "Age",
    "sex":       "Gender",
    "sexbirth1": "Sex recorded at birth",
    "race":      "Race",
    "racecen1":  "Race (detailed)",
    "hispanic":  "Hispanic origin",
    "ethnic":    "Ethnic ancestry",
    "sibs":      "Number of siblings",
    "marital":   "Marital status",
    "martype":   "Type of marriage",
    "widowed":   "Ever widowed",
    "divorce":   "Ever divorced",
    "childs":    "Number of children",
    "agekdbrn":  "Age at birth of first child",
    "region":    "Region of residence",
    "xnorcsiz":  "Size of place of residence",
    "uscitzn":   "US citizenship status",
    "born":      "Born in the United States",
    "migage":    "Age moved to current residence",
    "mighist":   "Migration history",
    "othlang":   "Language other than English spoken at home",
    "spklang":   "Fluency in other language",

    # -- Education -----------------------------------------------------
    "educ":      "Highest year of school completed",
    "degree":    "Highest degree",
    "major1":    "College major",

    # -- Employment --------------------------------------------------
    "wrkstat":   "Current employment status",
    "evwork":    "Ever worked as long as one year",
    "wrkslf":    "Self-employed or works for someone else",
    "wrkslffam": "Works in own family business or farm",
    "wrkgovt1":  "Employed by government",
    "indus10":   "Industry",
    "occ10":     "Occupation",
    "numemps":   "Number of employees at workplace",
    "localnum":  "Size of workplace",
    "hrs1":      "Hours worked last week",
    "weekswrk":  "Weeks worked last year",
    "partfull":  "Full-time or part-time work",
    "prestg10":  "Occupational prestige score",
    "sei10":     "Socioeconomic index score",
    "union":     "Labor union membership",
    "satjob":    "Job satisfaction",
    "yousup":    "Supervises other employees",

    # -- Income & class --------------------------------------------
    "coninc":    "Household income (dollars)",
    "conrinc":   "Personal income (dollars)",
    "class":     "Subjective social class",
    "satfin":    "Satisfaction with financial situation",
    "finrela":   "Financial situation compared to average American family",
    "parsol":    "Standard of living compared to parents",
    "kidssol":   "Expected standard of living for own children",
    "goodlife":  "Chance to improve standard of living",

    # -- Household --------------------------------------------------
    "hompop":    "Household size",
    "earnrs":    "Number of earners in household",
    "dwelling":  "Dwelling type",
    "dwelown":   "Home ownership",
    "famgen":    "Generations living in household",

    # -- Family / spouse ------------------------------------------
    "speduc":    "Spouse's education",
    "spdeg":     "Spouse's degree",
    "spwrksta":  "Spouse's employment status",
    "spocc10":   "Spouse's occupation",
    "spind10":   "Spouse's industry",
    "sppres10":  "Spouse's occupational prestige score",
    "spsei10":   "Spouse's socioeconomic index score",
    "hapmar":    "Happiness in marriage",

    # -- Parental / social-origin background ----------------------------
    "paeduc":    "Father's education",
    "maeduc":    "Mother's education",
    "paocc10":   "Father's occupation",
    "maocc10":   "Mother's occupation",
    "papres10":  "Father's occupational prestige score",
    "mapres10":  "Mother's occupational prestige score",
    "pasei10":   "Father's socioeconomic index score",
    "masei10":   "Mother's socioeconomic index score",
    "family16":  "Family situation growing up",
    "incom16":   "Family income growing up (relative)",
    "maborn":    "Mother born in the United States",
    "paborn":    "Father born in the United States",
    "granborn":  "Number of grandparents born outside the United States",
    "reg16":     "Region of residence at age 16",
    "res16":     "Type of place lived in at age 16",
    "mobile16":  "Geographic mobility since age 16",

    # -- Religiosity ----------------------------------------------
    "relig":     "Religious preference",
    "relig16":   "Religion raised in",
    "denom":     "Religious denomination",
    "attend":    "Frequency of attending religious services",
    "pray":      "Frequency of prayer",
    "relpersn":  "Self-rated religiosity",

    # -- Values ----------------------------------------------------
    "happy":     "General happiness",
    "getahead":  "Belief about how to get ahead in life",
    "richwork":  "Would keep working even if wealthy",
    "workhard":  "Value placed on hard work when raising children",

    # -- Lifestyle -------------------------------------------------
    "tvhours":   "Hours of TV watched per day",

    # -- Moral item (mirrors the one retained in the German pipeline) ----
    "spanking":  "Approval of spanking children",
}

available = [c for c in INCLUDE_VARIABLES if c in personas.columns]
missing = [c for c in INCLUDE_VARIABLES if c not in personas.columns]
if missing:
    print(f"Warning: {len(missing)} expected variables not found in extract: {missing}")


def _clean_scalar(value):
    """Render a survey value for the LLM prompt: round floats to cents to
    avoid binary-float artifacts (e.g. coninc's 46447.49999999999), then drop
    a trailing '.0' on whole numbers. Pass through everything else as-is."""
    if isinstance(value, float):
        value = round(value, 2)
        if value.is_integer():
            return str(int(value))
    return str(value)


selected = personas[available].rename(columns=INCLUDE_VARIABLES)
records = [
    {k: _clean_scalar(v) for k, v in row.items() if pd.notna(v)}
    for row in selected.to_dict(orient="records")
]

with open("../data/gss_personas.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

field_counts = [len(r) for r in records]
print(f"Exported {len(records)} records, {len(available)} possible fields "
      f"({min(field_counts)}-{max(field_counts)} populated per record, "
      f"avg {sum(field_counts)/len(field_counts):.1f})")

In [ ]:
# Values already arrive pre-cleaned via convert_categoricals=True in cell 1
# (unlike the old German CSV pipeline, which had literal numeric-code prefixes
# like '(2) ' to strip).
df_expanded = pd.DataFrame(records)


In [ ]:
# Separate numeric and categorical columns
numeric_cols = df_expanded.select_dtypes(include="number").columns.tolist()
cat_cols = [
    c for c in df_expanded.select_dtypes(exclude="number").columns
    if df_expanded[c].nunique() <= 10
]

# --- Numeric: histogram per column ---
if numeric_cols:
    df_num = df_expanded[numeric_cols].melt(var_name="Item", value_name="Value").dropna()
    df_num["Item"] = df_num["Item"].str[:60]
    g_num = sns.FacetGrid(df_num, col="Item", col_wrap=4, height=3, sharex=False, sharey=False)
    g_num.map(sns.histplot, "Value", bins=20)
    g_num.set_titles("{col_name}", size=7)
    g_num.figure.suptitle("Numeric Items", y=1.01)
    plt.tight_layout()
    plt.savefig("../img/distributions_numeric.png", dpi=150, bbox_inches="tight")
    plt.show()

# --- Categorical: horizontal bar chart per column ---
if cat_cols:
    df_cat = df_expanded[cat_cols].melt(var_name="Item", value_name="Value").dropna()
    df_cat["Item"] = df_cat["Item"].str[:60]

    g_cat = sns.FacetGrid(
        df_cat, col="Item", col_wrap=4, height=3, aspect=1.6,
        sharex=False, sharey=False,
    )
    def _countplot(data, **kwargs):
        order = data["Value"].value_counts().index
        sns.countplot(data=data, y="Value", order=order, **kwargs)
    g_cat.map_dataframe(_countplot)
    g_cat.set_titles("{col_name}", size=7)
    g_cat.set_axis_labels("Count", "")
    g_cat.figure.suptitle("Categorical Items (≤10 categories)", y=1.01)
    plt.tight_layout()
    plt.savefig("../img/distributions_categorical.png", dpi=150, bbox_inches="tight")
    plt.show()